## Twitter Sentiment Analysis

In [64]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer,LancasterStemmer
from wordcloud import WordCloud

import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from mlxtend.plotting import plot_confusion_matrix

## Importing the dataset

In [65]:
df = pd.read_csv(r"D:\Innomatics\Git_Uploads\twitter_sentiment_analysis\data\twitter_training.csv", header=None,
                 names=["tweet_id","entity","sentiment","tweet"])

In [66]:
df.head(5)

,tweet_id,entity,sentiment,tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [67]:
df.shape

(74682, 4)

In [68]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   tweet_id   74682 non-null  int64
 1   entity     74682 non-null  str  
 2   sentiment  74682 non-null  str  
 3   tweet      73996 non-null  str  
dtypes: int64(1), str(3)
memory usage: 11.4 MB


In [69]:
df.isnull().sum()

tweet_id       0
entity         0
sentiment      0
tweet        686
dtype: int64

In [70]:
df.dropna(subset="tweet",inplace=True)

In [71]:
df.isnull().sum()

tweet_id     0
entity       0
sentiment    0
tweet        0
dtype: int64

In [72]:
df["sentiment"].value_counts()

sentiment
Negative      22358
Positive      20655
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64

In [73]:
df[df["sentiment"] == "Irrelevant"]

,tweet_id,entity,sentiment,tweet
102,2418,Borderlands,Irrelevant,Appreciate the (sonic) concepts / praxis Valen...
103,2418,Borderlands,Irrelevant,Appreciate the (sound) concepts / practices th...
104,2418,Borderlands,Irrelevant,Evaluate the (sound) concepts / concepts of Va...
105,2418,Borderlands,Irrelevant,Appreciate the (sonic) concepts / praxis Valen...
106,2418,Borderlands,Irrelevant,Appreciate by the ( sonic ) electronic concept...
...,...,...,...,...
74035,9085,Nvidia,Irrelevant,This is all based on last quarter's earnings. ...
74036,9085,Nvidia,Irrelevant,Let's see how well they handle the next one wh...
74037,9085,Nvidia,Irrelevant,Good on them. This stuff all based on earnings...
74038,9085,Nvidia,Irrelevant,9 Good idea for them. This is all based on ear...


In [74]:
## Dropping the Irrevelant sentiment data as its not needed for our sentiment analysis

In [75]:
df = df[df["sentiment"] != "Irrelevant"].reset_index(drop=True)

In [76]:
df["sentiment"].value_counts()

sentiment
Negative    22358
Positive    20655
Neutral     18108
Name: count, dtype: int64

In [77]:
df_copy = df.copy()

In [78]:
df.drop(axis=0,columns="tweet_id",inplace= True)

In [79]:
## Assiging numbers for sentiment for faster computation

In [80]:
sentiment_num = {"Negative":0,"Neutral" : 1, "Positive":2}
df["sentiment"] = df["sentiment"].map(sentiment_num)
df["sentiment"].value_counts()

sentiment
0    22358
2    20655
1    18108
Name: count, dtype: int64

### Text Preprocessing

In [237]:
test = df["tweet"].sample().iloc[0]
test

'New Deal (Epic Store Free Games release schedule $1) -- latestblackfridaydeals.com / epic-store-fre....'

In [238]:
test = test.lower()

In [239]:
test = re.sub(r"http\S+|www\S+","",test) ## Removes Link
test = re.sub(r"@\w+|@+","",test) ## Removes Mentions
test = re.sub(r"<\S+","",test) ## Removes Tage
test = re.sub(r"\w\S+\.com\S+","",test) ## removes other .com text
test = re.sub(r"\w+\.\S+","",test)  ## removes site links
test =  re.sub(r"#","",test) ## Removes symbol
test

'new deal (epic store free games release schedule $1) --  / epic-store-'